In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import numpy as np

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

model.eval()

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
def calculate_perplexity(text):
    inputs = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(inputs, labels=inputs)
        loss = outputs.loss
        perplexity = torch.exp(loss)
    return perplexity.item()

In [4]:
from rdflib import Graph, Namespace, URIRef, Literal

In [ ]:
def process_ttl_file(input_file, output_file):
    # Create an RDF graph
    graph = Graph()

    # Parse the input TTL file
    graph.parse(input_file, format='ttl')

    # Define the namespace
    EX = Namespace("http://example.org/")

    # Iterate through all triples to find ex:Token subjects linked to other ex:Tokens
    tokens = set()
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef) and s.startswith(EX) and o.startswith(EX):
            if s != o:  # Ensure we're not linking a token to itself
                tokens.add((s, o))

    # Add new links with the label as the perplexity difference between nodes
    for s, o in tokens:
        perplexity_s = calculate_perplexity(str(s))
        perplexity_o = calculate_perplexity(str(o))
        perplexity_difference = abs(perplexity_s - perplexity_o)
        graph.add((s, EX.perplexity_diff, Literal(perplexity_difference)))

    # Serialize the graph to a new TTL file
    graph.serialize(destination=output_file, format='ttl')
    print(f"Graph successfully written to {output_file}")

In [11]:
# Define the input and output file names
input_ttl_file = 'test10.ttl'  # Replace with your actual input file name
output_ttl_file = 'perplexity_graph.ttl'

# Process the TTL file
process_ttl_file(input_ttl_file, output_ttl_file)

Graph successfully written to perplexity_graph.ttl
